# League of Legends Champion Draft AI Analyzer
### Riot Games API Multi-Game Draft Evaluation & Mobalytics Clone Model

This notebook connects directly to the official Riot Games API to predict on the **latest 10 games** (or the maximum amount possible):
1. **Dynamic Account Lookup**: Resolves Riot ID via `ACCOUNT-V1` across continental clusters (ASIA, AMERICAS, EUROPE) to retrieve the player's PUUID.
2. **Multi-Game Match History (`MATCH-V5`)**: Queries the player's recent matches (up to 10 on SEA, ASIA, etc.) and handles accounts with fewer games gracefully.
3. **Live Champion Catalog**: Queries Riot's official Data Dragon API for all 160+ champions on the active patch.
4. **AI Model Pipeline (`league_mobalytics_clone.joblib`)**: Reconstructs the 15 pre-game draft features (pick phases, champion win rates, keystone synergies, role interactions).
5. **Batch AI Predictions**: Iterates across each match draft, calculates win probabilities, Draft Edge (Δ), and Mobalytics Tier Badges.
6. **Multi-Game Summary & Accuracy**: Evaluates draft advantage vs actual match outcome across all analyzed games.

## Step 1: Configuration & Environment Setup
Set your target player, preferred region, and paste your active Riot API key below.

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from urllib.parse import quote
import requests
import numpy as np
import pandas as pd
import joblib

RIOT_API_KEY = "RGAPI-f8f59036-9dee-4dd7-bc04-c4817acc2792"
RIOT_API_KEY = RIOT_API_KEY.strip().strip('"\'').strip()

# Target player credentials
GAME_NAME = "Dinozo"   # Your in-game name
TAG_LINE  = "3141"     # Your tag (e.g. 3141, SG2, KR1, NA1)
REGION    = "sea"      # Preferred region ('sea', 'americas', 'asia', 'europe')

# Target number of recent games to analyze (predicts latest 10, or maximum available)
MAX_GAMES_TO_ANALYZE = 10

if 'PASTE-YOUR-ACTIVE-KEY-HERE' in RIOT_API_KEY or len(RIOT_API_KEY) < 30:
    print('=' * 80)
    print('⚠️  ACTION REQUIRED: YOU NEED TO PASTE YOUR ACTIVE RIOT API KEY!')
    print('=' * 80)
    print('1. Go to: https://developer.riotgames.com/')
    print('2. Log in and copy your active Development API Key.')
    print('3. Paste it above:  RIOT_API_KEY = "RGAPI-..."')
    print('4. Press Shift + Enter on this cell to load it into memory!')
    print('=' * 80)
else:
    print(f"Target Player:    {GAME_NAME}#{TAG_LINE} (Preferred Region: {REGION})")
    print(f"Games to Analyze: Up to {MAX_GAMES_TO_ANALYZE}")
    print(f"API Key Loaded:   {RIOT_API_KEY[:9]}...{RIOT_API_KEY[-4:]} (Length: {len(RIOT_API_KEY)})")


Target Player:    Dinozo#3141 (Preferred Region: sea)
Games to Analyze: Up to 10
API Key Loaded:   RGAPI-f8f...2792 (Length: 42)


## Step 2: Dynamic Player & Match History Lookup
Finds the player's PUUID and queries `MATCH-V5` across clusters to retrieve up to 10 recent match IDs.

In [2]:
if 'PASTE-YOUR-ACTIVE-KEY-HERE' in RIOT_API_KEY or len(RIOT_API_KEY) < 30:
    raise ValueError('Please paste your active Riot API key in Step 1 and run Step 1 (Shift+Enter) first!')

safe_name = quote(GAME_NAME.strip())
safe_tag  = quote(TAG_LINE.strip())
headers   = {"X-Riot-Token": RIOT_API_KEY}
primary_reg = REGION.strip().lower()

# 1. Resolve Player PUUID across valid Account-v1 shards ('asia', 'americas', 'europe')
# NOTE: Riot's Account-v1 API does NOT accept 'sea'. SEA accounts are resolved on 'asia'!
account_clusters = ["asia", "americas", "europe"] if primary_reg in ("sea", "asia") else ["americas", "europe", "asia"]
account = None
auth_error_resp = None

for reg in account_clusters:
    acc_url = f"https://{reg}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{safe_name}/{safe_tag}"
    resp = requests.get(acc_url, headers=headers, timeout=8)
    if resp.status_code == 200:
        account = resp.json()
        print(f"✓ Found account {GAME_NAME}#{TAG_LINE} on '{reg.upper()}' account cluster!")
        auth_error_resp = None
        break
    elif resp.status_code in (401, 403):
        auth_error_resp = resp
        break

if auth_error_resp is not None:
    code = auth_error_resp.status_code
    masked = f"{RIOT_API_KEY[:8]}...{RIOT_API_KEY[-4:]}" if len(RIOT_API_KEY) > 12 else RIOT_API_KEY
    print(f"Riot Gateway Response: {auth_error_resp.text}")
    if code == 403:
        raise RuntimeError(
            f"Error 403 (Forbidden): Riot rejected key '{masked}'.\n"
            "Riot development keys expire every 24 hours!\n"
            "Go to https://developer.riotgames.com/, log in, click 'REGENERATE API KEY', "
            "paste the fresh key into Step 1, and press Shift+Enter."
        )
    else:
        raise RuntimeError(
            f"Error 401 (Unauthorized): Riot rejected key '{masked}'.\n"
            "Please verify you copied the active key from https://developer.riotgames.com/ "
            "into Step 1 and pressed Shift+Enter."
        )

if not account:
    raise RuntimeError(f"Player '{GAME_NAME}#{TAG_LINE}' was not found. Please double-check spelling and tag.")

puuid = account["puuid"]
print(f"Player PUUID: {puuid}")

# 2. Find which match region has recent games for this player (request up to MAX_GAMES_TO_ANALYZE)
# Match-v5 DOES support 'sea.api.riotgames.com' for SG2/PH2/TW2/VN2/OC1 games
match_clusters = ["sea", "asia", "americas", "europe"] if primary_reg == "sea" else [primary_reg, "americas", "europe", "asia", "sea"]
seen_clusters = set()
match_clusters = [x for x in match_clusters if not (x in seen_clusters or seen_clusters.add(x))]

match_ids = []
active_match_region = match_clusters[0]

for reg in match_clusters:
    m_url = f"https://{reg}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?count={MAX_GAMES_TO_ANALYZE}"
    m_resp = requests.get(m_url, headers=headers, timeout=8)
    if m_resp.status_code == 200:
        res_list = m_resp.json()
        if isinstance(res_list, list) and len(res_list) > 0:
            match_ids = res_list
            active_match_region = reg
            print(f"✓ Found {len(match_ids)} recent matches on '{reg.upper()}' server!")
            break

# Determine matches to analyze (up to 10, or maximum available)
if match_ids:
    REGION = active_match_region
    matches_to_analyze = match_ids[:MAX_GAMES_TO_ANALYZE]
    print(f"\n✓ Found {len(match_ids)} available match(es) for {GAME_NAME}#{TAG_LINE}.")
    print(f"✓ Queueing {len(matches_to_analyze)} match(es) for AI prediction (Region: {REGION.upper()}):\n")
    for idx, mid in enumerate(matches_to_analyze, 1):
        print(f"   [{idx}] {mid}")
else:
    print(f"\n⚠️ Note: Account '{GAME_NAME}#{TAG_LINE}' exists, but has 0 recent League of Legends matches in history.")
    print("This happens if this Riot account has not played League of Legends recently or plays Valorant/TFT.")
    print("To demonstrate multi-match draft prediction, we will use sample active match IDs:")
    REGION = "sea" if primary_reg in ("sea", "asia") else "americas"
    if REGION == "sea":
        matches_to_analyze = [
            "SG2_315894001", "SG2_315894002", "SG2_315894003", "SG2_315894004", "SG2_315894005"
        ]
    else:
        matches_to_analyze = [
            "NA1_4987654321", "NA1_4987654322", "NA1_4987654323", "NA1_4987654324", "NA1_4987654325"
        ]
    for idx, mid in enumerate(matches_to_analyze, 1):
        print(f"   [{idx}] {mid} (sample)")


✓ Found account Dinozo#3141 on 'ASIA' account cluster!
Player PUUID: Nfu95QXabUJjxAziTfQzsK1cc95jeeRVeGjsgWf94GkW2jHBAbXS8WGJo5t9M7qe7caSgPQ-5gB5DQ
✓ Found 10 recent matches on 'SEA' server!

✓ Found 10 available match(es) for Dinozo#3141.
✓ Queueing 10 match(es) for AI prediction (Region: SEA):

   [1] SG2_178116877
   [2] SG2_178112708
   [3] SG2_176277729
   [4] SG2_168646635
   [5] SG2_168486597
   [6] SG2_168483294
   [7] SG2_168475918
   [8] SG2_168470680
   [9] SG2_168467469
   [10] SG2_168460882


## Step 3: Fetch Live Champions from Official Riot Games API
We query Riot's official **Data Dragon API** to retrieve the dynamic catalog of all 160+ champions.

In [3]:
# 1. Query latest patch version from Riot API
versions_url = 'https://ddragon.leagueoflegends.com/api/versions.json'
version_resp = requests.get(versions_url, timeout=10)
latest_version = version_resp.json()[0]

# 2. Query full champion catalog from Riot API
champ_url = f"https://ddragon.leagueoflegends.com/cdn/{latest_version}/data/en_US/champion.json"
champ_resp = requests.get(champ_url, timeout=10).json()
raw_champions = champ_resp['data']

# 3. Dynamically build champion mapping directly from Riot's API response
champion_id_to_name = {
    int(champ['key']): champ['name']
    for champ in raw_champions.values()
}
champion_id_to_data = {
    int(champ['key']): champ
    for champ in raw_champions.values()
}

def resolve_champ_name(cid):
    if cid == -1:
        return 'None (No Ban)'
    return champion_id_to_name.get(cid, f'Champion #{cid}')

print(f"✓ Dynamically loaded {len(champion_id_to_name)} champions from Riot API on Patch {latest_version}!")


✓ Dynamically loaded 173 champions from Riot API on Patch 16.20.1!


## Step 4: Load AI Model (`league_mobalytics_clone.joblib`)
Loads the trained pipeline bundle containing the calibrated random forest, feature schema, and win rate dictionaries.

In [4]:
# Required class definition to deserialize the notebook artifact
class MonotonicCalibratedForest:
    def __init__(self, base_model, calibrator):
        self.base_model = base_model
        self.calibrator = calibrator
        self.feature_importances_ = getattr(base_model, 'feature_importances_', None)

    def predict_proba(self, X):
        raw_p = self.base_model.predict_proba(X)[:, 1]
        cal_p = np.clip(self.calibrator.transform(raw_p), 0.01, 0.99)
        return np.column_stack([1 - cal_p, cal_p])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.50).astype(int)

# Register in __main__ so joblib unpickler resolves it
import __main__
__main__.MonotonicCalibratedForest = MonotonicCalibratedForest

model_filename = 'league_mobalytics_clone.joblib'
if os.path.exists(model_filename):
    loaded_pipeline = joblib.load(model_filename)
    print(f"✓ Loaded Pipeline: '{loaded_pipeline['model_name']}'")
    print(f"✓ Trained on:      {loaded_pipeline.get('training_rows', 514103):,} summoners")
    print(f"✓ Feature Schema:  {len(loaded_pipeline['feature_names'])} pre-game features")
    print(f"✓ Lookup Dicts:    {len(loaded_pipeline['champ_wr_dict'])} champions, {len(loaded_pipeline['keystone_wr_dict'])} keystones")
else:
    raise FileNotFoundError(f"⚠️ Artifact '{model_filename}' not found. Please ensure it is in the project folder.")


✓ Loaded Pipeline: 'league_mobalytics_clone'
✓ Trained on:      514,103 summoners
✓ Feature Schema:  15 pre-game features
✓ Lookup Dicts:    172 champions, 18 keystones


## Step 5: Draft Feature Engineering & Inference Engine
Defines the mathematical transformation pipeline that reconstructs the 15 features for each match draft and evaluates win probability and Mobalytics badges.

In [5]:
champ_wr = loaded_pipeline.get('champ_wr_dict', {})
keystone_wr = loaded_pipeline.get('keystone_wr_dict', {})
spell1_wr = loaded_pipeline.get('spell1_wr_dict', {})
spell2_wr = loaded_pipeline.get('spell2_wr_dict', {})
model = loaded_pipeline['model']
imputer = loaded_pipeline.get('imputer')
feature_names = loaded_pipeline['feature_names']

def get_mobalytics_badge(prob):
    delta = (prob - 0.50) * 100.0
    if delta >= 4.0:
        return f"{delta:+5.1f}%", '[Heavy Advantage]'
    elif delta >= 1.5:
        return f"{delta:+5.1f}%", '[Slight Advantage]'
    elif delta <= -4.0:
        return f"{delta:+5.1f}%", '[Heavy Disadvantage]'
    elif delta <= -1.5:
        return f"{delta:+5.1f}%", '[Slight Disadvantage]'
    else:
        return f"{delta:+5.1f}%", '[Even / Skill Match]'

def analyze_match_draft(match_json, target_puuid=None):
    """
    Extracts draft picks, bans, and player data from MATCH-V5 JSON,
    constructs the 15 pre-game features, and evaluates model predictions.
    """
    info = match_json.get('info', {})
    participants = info.get('participants', [])
    teams = info.get('teams', [])
    
    # 1. Parse Participants (Picks)
    picks_list = []
    target_player_info = None
    
    for p in participants:
        cid = p['championId']
        cname = p.get('championName', resolve_champ_name(cid))
        is_blue = (p['teamId'] == 100)
        side_str = "Blue" if is_blue else "Red"
        
        # Extract primary keystone safely
        keystone_id = 0
        perk_styles = p.get('perks', {}).get('styles', [])
        if perk_styles and 'selections' in perk_styles[0] and perk_styles[0]['selections']:
            keystone_id = perk_styles[0]['selections'][0].get('perk', 0)
            
        pick_entry = {
            'puuid': p.get('puuid'),
            'team_id': p['teamId'],
            'team_side': side_str,
            'player_name': f"{p.get('riotIdGameName', '')}#{p.get('riotIdTagline', '')}",
            'champion_id': cid,
            'champion_name': cname,
            'role': p.get('teamPosition', 'UNKNOWN'),
            'spell1': p.get('summoner1Id', 0),
            'spell2': p.get('summoner2Id', 0),
            'keystone_id': keystone_id,
            'won': bool(p.get('win', False)),
            'kills': p.get('kills', 0),
            'deaths': p.get('deaths', 0),
            'assists': p.get('assists', 0)
        }
        picks_list.append(pick_entry)
        
        if target_puuid and p.get('puuid') == target_puuid:
            target_player_info = pick_entry
            
    # Default to first participant if target player not found (e.g. sample matches)
    if target_player_info is None and picks_list:
        target_player_info = picks_list[0]
        
    # 2. Parse Bans
    bans_list = []
    for t in teams:
        side_str = "Blue" if t['teamId'] == 100 else "Red"
        for b in t.get('bans', []):
            cid = b['championId']
            bans_list.append({
                'team_side': side_str,
                'turn': b.get('pickTurn', 0),
                'champion_id': cid,
                'champion_name': resolve_champ_name(cid)
            })
            
    # 3. Feature Engineering
    blue_cids = [p['champion_id'] for p in picks_list if p['team_side'] == 'Blue']
    red_cids  = [p['champion_id'] for p in picks_list if p['team_side'] == 'Red']
    
    while len(blue_cids) < 5:
        blue_cids.append(1)
    while len(red_cids) < 5:
        red_cids.append(1)
        
    def cwr(cid):
        return champ_wr.get(cid, 0.50)
        
    # Pick phase win rates
    b1 = cwr(blue_cids[0])
    r1r2 = np.mean([cwr(red_cids[0]), cwr(red_cids[1])])
    b2b3 = np.mean([cwr(blue_cids[1]), cwr(blue_cids[2])])
    r3r4 = np.mean([cwr(red_cids[2]), cwr(red_cids[3])])
    b4b5 = np.mean([cwr(blue_cids[3]), cwr(blue_cids[4])])
    r5 = cwr(red_cids[4])

    raw_p1 = b1 - r1r2
    raw_p2 = b2b3 - r3r4
    raw_p3 = b4b5 - r5

    blue_team_wr = np.mean([cwr(c) for c in blue_cids[:5]])
    red_team_wr = np.mean([cwr(c) for c in red_cids[:5]])
    raw_team_adv = blue_team_wr - red_team_wr

    # Perspective: evaluate for the target player's team
    is_target_blue = (target_player_info['team_side'] == 'Blue')
    adv_p1 = raw_p1 if is_target_blue else -raw_p1
    adv_p2 = raw_p2 if is_target_blue else -raw_p2
    adv_p3 = raw_p3 if is_target_blue else -raw_p3
    team_adv = raw_team_adv if is_target_blue else -raw_team_adv
    my_team_wr = blue_team_wr if is_target_blue else red_team_wr
    enemy_team_wr = red_team_wr if is_target_blue else blue_team_wr

    my_c = target_player_info['champion_id']
    my_champ_wr = cwr(my_c)
    # Opponent lane champion (same role or counterpart)
    enemy_c = red_cids[2] if is_target_blue else blue_cids[2]
    enemy_champ_wr = cwr(enemy_c)
    champ_adv = my_champ_wr - enemy_champ_wr

    log_mastery = np.log1p(50000.0)
    mastery_champ_synergy = log_mastery * my_champ_wr
    
    s1_w = spell1_wr.get(target_player_info.get('spell1', 4), 0.50)
    s2_w = spell2_wr.get(target_player_info.get('spell2', 14), 0.50)
    spell_synergy = 0.5 * (s1_w + s2_w)
    
    k_id = target_player_info.get('keystone_id', 8112)
    keystone_val = keystone_wr.get(k_id, 0.50)
    
    total_synergy = champ_adv + team_adv
    draft_momentum = 0.20 * adv_p1 + 0.35 * adv_p2 + 0.45 * adv_p3

    feature_dict = {
        'champ_adv': champ_adv,
        'my_champ_wr': my_champ_wr,
        'team_adv': team_adv,
        'my_team_wr': my_team_wr,
        'enemy_team_wr': enemy_team_wr,
        'adv_p1': adv_p1,
        'adv_p2': adv_p2,
        'adv_p3': adv_p3,
        'draft_momentum': draft_momentum,
        'total_synergy': total_synergy,
        'log_mastery': log_mastery,
        'mastery_champ_synergy': mastery_champ_synergy,
        'spell_synergy': spell_synergy,
        'keystone_wr': keystone_val,
        'is_blue': 1 if is_target_blue else 0
    }

    X_df = pd.DataFrame([feature_dict])[feature_names]
    X_mat = imputer.transform(X_df) if imputer is not None else X_df.values
    player_win_prob = float(model.predict_proba(X_mat)[:, 1][0])
    delta_str, badge = get_mobalytics_badge(player_win_prob)
    actual_won = target_player_info['won']
    predicted_win = (player_win_prob >= 0.50)
    
    return {
        'target_player': target_player_info,
        'is_target_blue': is_target_blue,
        'player_win_prob': player_win_prob,
        'draft_edge_delta': delta_str,
        'mobalytics_badge': badge,
        'actual_won': actual_won,
        'prediction_correct': (predicted_win == actual_won),
        'blue_cids': blue_cids,
        'red_cids': red_cids,
        'picks': picks_list,
        'bans': bans_list,
        'game_duration': info.get('gameDuration', 0)
    }

print("✓ Feature engineering and prediction engine initialized!")


✓ Feature engineering and prediction engine initialized!


## Step 6: Ingest & Predict on Latest 10 Games (or Max Available)
Iterates through all queued matches, fetches official `MATCH-V5` payloads, extracts draft compositions, and computes win probability for each game.

In [6]:
analyzed_records = []
total_to_run = len(matches_to_analyze)

print(f"Initiating batch draft prediction on {total_to_run} match(es) from {REGION.upper()}...\n")

for idx, match_id in enumerate(matches_to_analyze, 1):
    m_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    resp = requests.get(m_url, headers=headers, timeout=12)
    
    if resp.status_code != 200:
        print(f"[{idx}/{total_to_run}] ⚠️ Error {resp.status_code} querying {match_id}: {resp.text[:60]}. Skipping.")
        continue
        
    m_json = resp.json()
    res = analyze_match_draft(m_json, target_puuid=puuid)
    p_info = res['target_player']
    
    outcome_str = "WIN" if res['actual_won'] else "LOSS"
    correct_str = "✓ Correct" if res['prediction_correct'] else "✗ Inverted"
    
    analyzed_records.append({
        'Match #': idx,
        'Match ID': match_id,
        'Champion': p_info['champion_name'],
        'Role': p_info['role'],
        'Side': p_info['team_side'],
        'KDA': f"{p_info['kills']}/{p_info['deaths']}/{p_info['assists']}",
        'Win Prob': f"{res['player_win_prob'] * 100:.1f}%",
        'Draft Edge (Δ)': res['draft_edge_delta'],
        'Mobalytics Badge': res['mobalytics_badge'],
        'Actual Result': outcome_str,
        'Model Accurate?': correct_str,
        '_raw': res
    })
    
    print(f"[{idx:>2}/{total_to_run}] {match_id:<16} | {p_info['champion_name']:<12} ({p_info['role']:<7}) | {p_info['team_side']:<4} | Win Prob: {res['player_win_prob']*100:5.1f}% | Draft Edge: {res['draft_edge_delta']:<7} | Result: {outcome_str:<4} ({correct_str})")
    time.sleep(0.12)  # Respect Riot API rate limits (20 req / 1 sec)

print(f"\n✓ Completed batch draft prediction for {len(analyzed_records)} game(s)!")


Initiating batch draft prediction on 10 match(es) from SEA...

[ 1/10] SG2_178116877    | Nunu         (UTILITY) | Blue | Win Prob:  54.2% | Draft Edge:  +4.2%  | Result: WIN  (✓ Correct)
[ 2/10] SG2_178112708    | Talon        (UTILITY) | Blue | Win Prob:  49.8% | Draft Edge:  -0.2%  | Result: WIN  (✗ Inverted)
[ 3/10] SG2_176277729    | Shaco        (MIDDLE ) | Red  | Win Prob:  45.8% | Draft Edge:  -4.2%  | Result: WIN  (✗ Inverted)
[ 4/10] SG2_168646635    | Nunu         (JUNGLE ) | Red  | Win Prob:  49.8% | Draft Edge:  -0.2%  | Result: WIN  (✗ Inverted)
[ 5/10] SG2_168486597    | Nunu         (UTILITY) | Blue | Win Prob:  47.1% | Draft Edge:  -2.9%  | Result: LOSS (✓ Correct)
[ 6/10] SG2_168483294    | Nunu         (JUNGLE ) | Red  | Win Prob:  56.5% | Draft Edge:  +6.5%  | Result: LOSS (✗ Inverted)
[ 7/10] SG2_168475918    | Shaco        (UTILITY) | Red  | Win Prob:  54.3% | Draft Edge:  +4.3%  | Result: LOSS (✗ Inverted)
[ 8/10] SG2_168470680    | Nunu         (JUNGLE ) | Red  

## Step 7: Consolidated Multi-Game Draft Evaluation & Summary Table
Aggregates the predictions across all evaluated games, showing champion played, lane role, draft edge, Mobalytics badge, and prediction accuracy.

In [7]:
if analyzed_records:
    df_history = pd.DataFrame(analyzed_records).drop(columns=['_raw'])
    print("=" * 110)
    print(f"  MULTI-GAME DRAFT ADVANTAGE SUMMARY FOR {GAME_NAME}#{TAG_LINE} (LATEST {len(analyzed_records)} GAMES)")
    print("=" * 110)
    display(df_history)

    # Calculate Summary Statistics
    total_games = len(analyzed_records)
    wins = sum(1 for r in analyzed_records if r['Actual Result'] == 'WIN')
    losses = total_games - wins
    actual_wr = (wins / total_games) * 100 if total_games > 0 else 0
    correct_preds = sum(1 for r in analyzed_records if 'Correct' in r['Model Accurate?'])
    pred_acc = (correct_preds / total_games) * 100 if total_games > 0 else 0
    
    # Draft Edge Breakdown
    adv_games = [r for r in analyzed_records if '+' in r['Draft Edge (Δ)']]
    disadv_games = [r for r in analyzed_records if '-' in r['Draft Edge (Δ)']]
    even_games = [r for r in analyzed_records if '+' not in r['Draft Edge (Δ)'] and '-' not in r['Draft Edge (Δ)']]
    
    adv_wins = sum(1 for r in adv_games if r['Actual Result'] == 'WIN')
    disadv_wins = sum(1 for r in disadv_games if r['Actual Result'] == 'WIN')
    
    adv_wr = (adv_wins / len(adv_games) * 100) if adv_games else 0.0
    disadv_wr = (disadv_wins / len(disadv_games) * 100) if disadv_games else 0.0
    
    print("\n" + "-" * 75)
    print(f"📊 SUMMARY PERFORMANCE METRICS ({total_games} GAMES ANALYZED):")
    print("-" * 75)
    print(f"• Actual Match Record:         {wins} Wins - {losses} Losses ({actual_wr:.1f}% Win Rate)")
    print(f"• AI Model Prediction Accuracy: {pred_acc:.1f}% ({correct_preds}/{total_games} games correctly classified)")
    print(f"• Games with Draft Advantage:   {len(adv_games)} games -> Actual Win Rate: {adv_wr:.1f}%")
    print(f"• Games with Draft Deficit:     {len(disadv_games)} games -> Actual Win Rate: {disadv_wr:.1f}%")
    print(f"• Even / Skill Match Games:     {len(even_games)} games")
    print("-" * 75)
else:
    print("⚠️ No matches were successfully analyzed. Please check API key status and region.")


  MULTI-GAME DRAFT ADVANTAGE SUMMARY FOR Dinozo#3141 (LATEST 10 GAMES)


,Match #,Match ID,Champion,Role,Side,KDA,Win Prob,Draft Edge (Δ),Mobalytics Badge,Actual Result,Model Accurate?
0,1,SG2_178116877,Nunu,UTILITY,Blue,12/8/9,54.2%,+4.2%,[Heavy Advantage],WIN,✓ Correct
1,2,SG2_178112708,Talon,UTILITY,Blue,8/6/8,49.8%,-0.2%,[Even / Skill Match],WIN,✗ Inverted
2,3,SG2_176277729,Shaco,MIDDLE,Red,14/6/8,45.8%,-4.2%,[Heavy Disadvantage],WIN,✗ Inverted
3,4,SG2_168646635,Nunu,JUNGLE,Red,17/14/19,49.8%,-0.2%,[Even / Skill Match],WIN,✗ Inverted
4,5,SG2_168486597,Nunu,UTILITY,Blue,0/5/9,47.1%,-2.9%,[Slight Disadvantage],LOSS,✓ Correct
5,6,SG2_168483294,Nunu,JUNGLE,Red,5/11/11,56.5%,+6.5%,[Heavy Advantage],LOSS,✗ Inverted
6,7,SG2_168475918,Shaco,UTILITY,Red,17/13/16,54.3%,+4.3%,[Heavy Advantage],LOSS,✗ Inverted
7,8,SG2_168470680,Nunu,JUNGLE,Red,15/6/13,58.8%,+8.8%,[Heavy Advantage],LOSS,✗ Inverted
8,9,SG2_168467469,Nunu,JUNGLE,Blue,6/11/6,49.8%,-0.2%,[Even / Skill Match],WIN,✗ Inverted
9,10,SG2_168460882,Shaco,UTILITY,Blue,7/16/21,47.1%,-2.9%,[Slight Disadvantage],WIN,✗ Inverted



---------------------------------------------------------------------------
📊 SUMMARY PERFORMANCE METRICS (10 GAMES ANALYZED):
---------------------------------------------------------------------------
• Actual Match Record:         6 Wins - 4 Losses (60.0% Win Rate)
• AI Model Prediction Accuracy: 20.0% (2/10 games correctly classified)
• Games with Draft Advantage:   4 games -> Actual Win Rate: 25.0%
• Games with Draft Deficit:     6 games -> Actual Win Rate: 83.3%
• Even / Skill Match Games:     0 games
---------------------------------------------------------------------------


## Step 8: Deep-Dive Match Draft Inspector
Inspects the full draft of any game in your match history (e.g. Match #1, the latest game), displaying all 10 picks, 10 bans, and tactical composition scores (Engage, Poke, Scaling, Teamfight, AD/AP ratio).

In [13]:
if analyzed_records:
    # Select which match to inspect (default: Match 1, latest match)
    inspect_index = 6
    inspected = analyzed_records[inspect_index]
    raw_info = inspected['_raw']
    
    print(f"=== DETAILED DRAFT INSPECTION: MATCH {inspected['Match ID']} (Match #{inspected['Match #']}) ===")
    print(f"Player: {GAME_NAME}#{TAG_LINE} playing {inspected['Champion']} ({inspected['Role']}) on {inspected['Side']} Team")
    print(f"Draft Edge: {inspected['Draft Edge (Δ)']} {inspected['Mobalytics Badge']} | Result: {inspected['Actual Result']}\n")
    
    # 1. Picks Table
    df_picks = pd.DataFrame(raw_info['picks'])
    display(df_picks[['team_side', 'player_name', 'role', 'champion_name', 'champion_id', 'kills', 'deaths', 'assists', 'won']])
    
    # 2. Bans Table
    df_bans = pd.DataFrame(raw_info['bans'])
    if not df_bans.empty:
        print("\n=== CHAMPION BANS ===")
        display(df_bans[['team_side', 'turn', 'champion_name']])
        
    # 3. Tactical Scores
    def score_team_tactics(cids):
        engages, disengages, pokes, scalings, teamfights, ads, aps = [], [], [], [], [], [], []
        for cid in cids:
            c_info = champion_id_to_data.get(cid, {})
            tags = c_info.get('tags', [])
            inf = c_info.get('info', {})
            att, df, mg = inf.get('attack', 5), inf.get('defense', 5), inf.get('magic', 5)
            
            e = 4.0 + (4.0 if 'Tank' in tags else 0) + (2.0 if 'Fighter' in tags else 0)
            d = 3.0 + (4.0 if 'Support' in tags else 0) + (2.0 if 'Mage' in tags else 0)
            p = 3.0 + (3.0 if ('Mage' in tags or 'Marksman' in tags) else 0)
            s = 5.0 + (2.5 if ('Marksman' in tags or 'Mage' in tags) else 0)
            tf = 4.0 + (3.5 if ('Mage' in tags or 'Tank' in tags) else 0) + (2.0 if 'Marksman' in tags else 0)
            
            engages.append(min(e, 10.0))
            disengages.append(min(d, 10.0))
            pokes.append(min(p, 10.0))
            scalings.append(min(s, 10.0))
            teamfights.append(min(tf, 10.0))
            tot = max(att + mg, 1)
            ads.append(att / tot)
            aps.append(mg / tot)
            
        return {
            'Engage': round(float(np.mean(engages)), 1),
            'Disengage': round(float(np.mean(disengages)), 1),
            'Poke': round(float(np.mean(pokes)), 1),
            'Scaling': round(float(np.mean(scalings)), 1),
            'Teamfight': round(float(np.mean(teamfights)), 1),
            'AD%': f"{round(float(np.mean(ads)) * 100, 0):.0f}%",
            'AP%': f"{round(float(np.mean(aps)) * 100, 0):.0f}%",
        }

    tactics = {
        'Blue Team': score_team_tactics(raw_info['blue_cids']),
        'Red Team':  score_team_tactics(raw_info['red_cids'])
    }
    df_tactics = pd.DataFrame(tactics)
    print("\n=== 5v5 TACTICAL COMPOSITION METRICS ===")
    display(df_tactics)


=== DETAILED DRAFT INSPECTION: MATCH SG2_168475918 (Match #7) ===
Player: Dinozo#3141 playing Shaco (UTILITY) on Red Team
Draft Edge:  +4.3% [Heavy Advantage] | Result: LOSS



,team_side,player_name,role,champion_name,champion_id,kills,deaths,assists,won
0,Blue,beholdthewolven#7499,TOP,Warwick,19,11,5,11,True
1,Blue,vit#ind,JUNGLE,AurelionSol,136,8,14,12,True
2,Blue,Ashi#Dncer,MIDDLE,Akali,84,12,7,14,True
3,Blue,AVoq7234#7234,BOTTOM,Karthus,30,15,16,16,True
4,Blue,Udyr is boring#hahah,UTILITY,Rengar,107,10,8,8,True
5,Red,Aatrosaurus#2005,TOP,Aatrox,266,9,7,3,False
6,Red,Elpilyo25#6179,JUNGLE,MonkeyKing,62,10,9,8,False
7,Red,Duck Dark#SG2,MIDDLE,Alistar,12,5,11,13,False
8,Red,Sela#5069,BOTTOM,Samira,360,9,16,14,False
9,Red,Dinozo#3141,UTILITY,Shaco,35,17,13,16,False



=== 5v5 TACTICAL COMPOSITION METRICS ===


,Blue Team,Red Team
Engage,5.6,6.4
Disengage,3.8,3.8
Poke,4.2,3.6
Scaling,6.0,5.5
Teamfight,6.1,5.8
AD%,46%,67%
AP%,54%,33%


## Step 9: API Limitations & Objective vs. Inferred Data
| Dimension | Source | Status / Limitation |
| :--- | :--- | :--- |
| **Champion Catalog & Names** | Official Riot Data Dragon API | **Objective API Data** (Updated every patch) |
| **Champion Picks & Bans** | Official `MATCH-V5` | **Objective API Data** |
| **Match Outcome (Win/Loss)** | Official `MATCH-V5` | **Objective API Data** |
| **Player Roles (`teamPosition`)** | Official `MATCH-V5` | **Inferred by Riot Post-Game Heuristic** |
| **Chronological Pick Order** | Not in `MATCH-V5` | **Inferred** (`participants` array is ordered 1..10, not B1->R1/R2) |
| **Draft Edge (Δ) & Badges** | AI Model (`league_mobalytics_clone.joblib`) | **Analytical Prediction** |